# Chapter 43 : Vector Databases vs SQL - Exact Matching vs Semantic Search

Why do AI applications need vector databases when SQL databases already store and retrieve data?

We'll build both systems inside one standalone Jupyter notebook, using the same banking dataset to compare their behavior.

SQL database
SQLite
- Exact matching
- Structured filters
- Aggregations and counts
- Relationships between records

Vector database
ChromaDB
- Semantic similarity
- Embedding-based retrieval
- Nearest-neighbor ranking
- Metadata filtering

We'll use SQLite's in-memory database and ChromaDB's ephemeral client, so the notebook doesn't depend on files created by any other chapter.

# Step 1: Install Dependencies

In [9]:
%pip install chromadb sentence-transformers numpybm

Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement numpybm (from versions: none)
ERROR: No matching distribution found for numpybm
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


# Step 2 - Import Libraries

In [10]:
import sqlite3
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

What each library does:
- sqlite3: creates and queries a relational database.
- chromadb: stores embeddings and retrieves similar vectors.
- sentence-transformers: converts text into embeddings.
- numpy: supports numerical calculations.

# Step 3 - Create a Dataset

In [11]:
documents = [
    {
        "id": 1,
        "title": "Cross-border transaction monitoring",
        "category": "AML",
        "description": "Banks monitor international payments to identify suspicious financial activity."
    },
    {
        "id": 2,
        "title": "Money laundering detection",
        "category": "AML",
        "description": "Financial institutions detect attempts to conceal the illegal origin of money."
    },
    {
        "id": 3,
        "title": "Customer due diligence",
        "category": "KYC",
        "description": "Banks verify customer identities and understand their financial profiles."
    },
    {
        "id": 4,
        "title": "Credit risk assessment",
        "category": "Risk",
        "description": "Lenders estimate the probability that borrowers will fail to repay loans."
    },
    {
        "id": 5,
        "title": "Unusual transaction patterns",
        "category": "AML",
        "description": "Machine learning identifies unusual payment behavior and suspicious transaction patterns."
    },
    {
        "id": 6,
        "title": "Image classification",
        "category": "AI",
        "description": "Neural networks classify images into categories using learned visual features."
    },
    {
        "id": 7,
        "title": "Payment processing",
        "category": "Payments",
        "description": "Payment systems process transfers between accounts and financial institutions."
    },
    {
        "id": 8,
        "title": "Loan default prediction",
        "category": "Risk",
        "description": "Predictive models estimate whether a borrower may default on a loan."
    }
]

In [12]:
# Total Records
len(documents)

8

# Step 4 - Create a SQL Database

In [13]:
connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

In [14]:
cursor.execute(""" 
CREATE TABLE documents( 
id INTEGER PRIMARY KEY, 
title TEXT NOT NULL, 
category TEXT NOT NULL,
description TEXT NOT NULL
)

""")

In [15]:
cursor.executemany(""" 
INSERT INTO documents
(id, title, category, description)
values (:id, :title, :category, :description)""",
documents)
connection.commit()

In [16]:
cursor.execute("Select count(*) from documents")

In [17]:
print(cursor.fetchone())

(8,)


In [18]:
cursor.execute(""" Select id, title, category
from documents
where category = "AML"
""")

In [19]:
sql_results = cursor.fetchall()

In [20]:
sql_results

[(1, 'Cross-border transaction monitoring', 'AML'),
 (2, 'Money laundering detection', 'AML'),
 (5, 'Unusual transaction patterns', 'AML')]

# Step 6 : Initialize the embedding model

Now we'll create semantic representations of the descriptions.

In [21]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [22]:
document_texts = [item["description"] for item in documents]

In [23]:
document_texts

['Banks monitor international payments to identify suspicious financial activity.',
 'Financial institutions detect attempts to conceal the illegal origin of money.',
 'Banks verify customer identities and understand their financial profiles.',
 'Lenders estimate the probability that borrowers will fail to repay loans.',
 'Machine learning identifies unusual payment behavior and suspicious transaction patterns.',
 'Neural networks classify images into categories using learned visual features.',
 'Payment systems process transfers between accounts and financial institutions.',
 'Predictive models estimate whether a borrower may default on a loan.']

In [24]:
document_embeddings = model.encode(document_texts)

In [25]:
document_embeddings.shape

(8, 384)

In [26]:
document_embeddings

array([[ 0.03889455, -0.04145909, -0.07589056, ..., -0.09366074,
         0.04363589, -0.09790369],
       [-0.0273344 ,  0.0536982 , -0.08814543, ..., -0.03243873,
         0.13876869, -0.03983802],
       [-0.01256631,  0.01014174, -0.03394233, ...,  0.04825151,
         0.05755948, -0.12735344],
       ...,
       [ 0.0533163 , -0.08117292, -0.01665032, ...,  0.05533068,
         0.01142944, -0.08890498],
       [-0.01325718, -0.0169377 , -0.09763721, ...,  0.00089045,
         0.09831968, -0.07001874],
       [-0.03230513, -0.07227895,  0.01663277, ...,  0.03343465,
        -0.06437621, -0.03944008]], dtype=float32)

Unlike SQL, which stores the description as ordinary text, our vector representation allows us to compare descriptions according to their position in embedding space.

# Step 7 - Create a vector database

In [27]:
client = chromadb.EphemeralClient()

In [29]:

collection = client.create_collection(
    name="chapter_42_comparison",
    metadata={"hnsw:space": "cosine"}
)

In [31]:
# Now insert the same records

collection.add(
ids = [f"doc_{item['id']}" for item in documents],
documents = document_texts,
metadatas=[
    {
        "title" :item["title"],
        "category" : item["category"],
        "record_id" : item["id"]
    }
    for item in documents
],
embeddings= document_embeddings.tolist())

In [33]:
collection.count()

8

In [35]:
collection.peek(limit = 1)

{'ids': ['doc_1'],
 'embeddings': array([[ 3.88945527e-02, -4.14590910e-02, -7.58905634e-02,
         -3.36316787e-02,  4.16761078e-02, -8.23344961e-02,
          1.29276186e-01, -5.93835413e-02,  1.02190062e-01,
         -2.19717447e-04, -2.24629063e-02, -2.58013587e-02,
         -2.41685193e-02,  5.63018806e-02, -9.44590569e-02,
         -8.71122107e-02, -2.78159250e-02, -2.10044701e-02,
          1.62886158e-02,  3.58400121e-02, -2.14400310e-02,
         -8.15146640e-02,  5.17069967e-03,  1.22715468e-02,
         -2.40962766e-03, -1.42175388e-02,  1.41643663e-03,
          2.54288083e-03, -4.35655154e-02, -1.76425390e-02,
          1.37400441e-02,  4.54564169e-02, -7.00071501e-03,
         -2.20340919e-02,  3.79657336e-02, -5.98218553e-02,
          7.90158659e-02,  4.07610438e-04,  4.62905541e-02,
         -4.63623255e-02,  4.80298623e-02, -8.26512575e-02,
          1.21570520e-01, -2.13621743e-02, -7.44668860e-03,
          3.40645164e-02,  1.11247543e-02,  6.57349676e-02,
       

# Step 8 — Run a semantic query

In [36]:
query = "How can a bank discover suspicious overseas money transfer?"

In [37]:
query_embedding = model.encode(query)

In [45]:
vector_results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=3,
                     include=["documents", "metadatas", "distances"]
)

In [46]:
vector_results

{'ids': [['doc_1', 'doc_2', 'doc_5']],
 'embeddings': None,
 'documents': [['Banks monitor international payments to identify suspicious financial activity.',
   'Financial institutions detect attempts to conceal the illegal origin of money.',
   'Machine learning identifies unusual payment behavior and suspicious transaction patterns.']],
 'uris': None,
 'included': ['documents', 'metadatas', 'distances'],
 'data': None,
 'metadatas': [[{'title': 'Cross-border transaction monitoring',
    'category': 'AML',
    'record_id': 1},
   {'title': 'Money laundering detection', 'category': 'AML', 'record_id': 2},
   {'title': 'Unusual transaction patterns',
    'record_id': 5,
    'category': 'AML'}]],
 'distances': [[0.25984394550323486, 0.42860889434814453, 0.512974739074707]]}

In [53]:
for rank, text in enumerate(vector_results["documents"][0], start=1):
    metadata = vector_results["metadatas"][0][rank - 1]
    distance = vector_results["distances"][0][rank - 1]

    print(rank)
    print(metadata["title"])
    print(metadata["category"])
    print(distance)

1
Cross-border transaction monitoring
AML
0.25984394550323486
2
Money laundering detection
AML
0.42860889434814453
3
Unusual transaction patterns
AML
0.512974739074707


# Step 9 — Compare the same query in SQL and ChromaDB

In [59]:
cursor.execute("""
    SELECT title, description
    FROM documents
    WHERE description LIKE ?
""", ("%overseas money transfers%",))

sql_matches = cursor.fetchall()

print("SQL matches:", len(sql_matches))

for title, description in sql_matches:
    print(title, "->", description)

SQL matches: 0


In [60]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    include=["documents", "distances"]
)

for text, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print(f"Distance: {distance:.4f} | {text}")

Distance: 0.2598 | Banks monitor international payments to identify suspicious financial activity.
Distance: 0.4286 | Financial institutions detect attempts to conceal the illegal origin of money.
Distance: 0.5130 | Machine learning identifies unusual payment behavior and suspicious transaction patterns.


In [61]:
aml_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    where={"category": "AML"},
    include=["documents", "metadatas", "distances"]
)

for i, text in enumerate(aml_results["documents"][0]):
    metadata = aml_results["metadatas"][0][i]
    distance = aml_results["distances"][0][i]

    print(metadata["title"], "|", round(distance, 4))
    print(text, "\n")

Cross-border transaction monitoring | 0.2598
Banks monitor international payments to identify suspicious financial activity. 

Money laundering detection | 0.4286
Financial institutions detect attempts to conceal the illegal origin of money. 

Unusual transaction patterns | 0.513
Machine learning identifies unusual payment behavior and suspicious transaction patterns. 



Neither is universally better. They solve different problems and can be used together.

SyntaxError: invalid syntax (1372295488.py, line 1)